[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-processing/N2N-SPRVS/blob/main/N2N-SPRVS.ipynb)

# Denoise (epfl_20nm)

In [ ]:
%run _params_.ipynb

In [ ]:
gpu_id = 0

In [ ]:
N = 1  # >=0 Number of generated tomograms used for training the denoiser

In [ ]:
!nvidia-smi

## Packages

In [ ]:
import json
from collections import namedtuple

In [ ]:
Args = namedtuple("args", ["X", "similar_X"])
args = Args("/home/vruiz/Tomograms/tomograms/Experimento_saltoZ/epfl_20nm.mrc", "001.mrc")

In [ ]:
def generate_cryocare_config(N):
    even_tomograms = [args.X]
    odd_tomograms = []
    for i in range(1,N+1):
        filename = f"{i:03d}.mrc"
        # Sort into even or odd lists
        if i % 2 == 0:
            even_tomograms.append(filename)
        else:
            odd_tomograms.append(filename)

    # Construct the final configuration dictionary
    config = {
        "even": even_tomograms,
        "odd": odd_tomograms,
        "mask": [""],
        "patch_shape": [16, 16, 16],
        "num_slices": 800,
        "split": 0.9,
        "tilt_axis": "Y",
        "n_normalization_samples": 200,
        "path": "./data_SPRVS",
        "overwrite": "True"
    }

    return config

_ = generate_cryocare_config(N)

with open("train_data_config__SPRVS.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat train_data_config__SPRVS.json

In [ ]:
%%bash
cryoCARE_extract_train_data.py --conf train_data_config__SPRVS.json

In [ ]:
_ = {
  "train_data": "./data_SPRVS",
  "epochs": 50,
  "steps_per_epoch": 200,
  "batch_size": 16,
  "unet_kern_size": 3,
  "unet_n_depth": 3,
  "unet_n_first": 16,
  "learning_rate": 0.0004,
  "model_name": "model_SPRVS",
  "path": "./",
  "gpu_id": [gpu_id]
}

with open("train_config__SPRVS.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat train_config__SPRVS.json

In [ ]:
%%bash
cryoCARE_train.py --conf train_config__SPRVS.json

In [ ]:
def generate_cryocare_config(N):
    even_tomograms = [args.X]
    odd_tomograms = []
    for i in range(1,N+1):
        filename = f"{i:03d}.mrc"
        # Sort into even or odd lists
        if i % 2 == 0:
            even_tomograms.append(filename)
        else:
            odd_tomograms.append(filename)

    # Construct the final configuration dictionary
    config = {
        "path": "./model_SPRVS.tar.gz",
        "even": even_tomograms,
        "odd": even_tomograms,
        "n_tiles": [2,2,2],
        "output": "denoised_vol_SPRVS",
        "overwrite": "True",
        "gpu_id": [gpu_id]
    }

    return config

_ = generate_cryocare_config(N)

with open("predict_config__SPRVS.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat predict_config__SPRVS.json

In [ ]:
%%bash
rm denoised_vol_SPRVS/*
cryoCARE_predict.py --conf predict_config__SPRVS.json || true

In [ ]:
!ls -l denoised_vol_SPRVS/*